# 05 - Generate evaluation outputs + blind pair files (Task 6)

Attach the saved outputs of notebooks 01 (SFT), 02 (DPO-HH), 03 (DPO-UF) and 04 (LoRA). Accelerator: **GPU T4 x2**. ~0.5-1h.

All Qwen models use the same prompt template and identical decoding: greedy, repetition_penalty 1.1, max_new_tokens 512.

In [ ]:
REPO_URL = "https://github.com/IsrarAliKhan/small-model-alignment-starter.git"  # edit if your fork lives elsewhere
%cd /kaggle/working
!git clone -q {REPO_URL} || git -C small-model-alignment-starter pull -q
%cd /kaggle/working/small-model-alignment-starter
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib pytest "peft>=0.11.0"
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("CUDA devices:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import glob

def find(pattern):
    """Locate a file/dir inside attached Kaggle inputs (outputs of earlier notebooks)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Not found: {pattern}. Attach the saved output of the notebook that produced it."
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
SFT = find("outputs/sft/best")
DPO_HH = find("outputs/dpo_hh/best")
DPO_UF = find("outputs/dpo_uf/best")
LORA = find("outputs/lora_persona/adapter")

Sanity check (3 prompts per model): checkpoint path, tokenizer, template, EOS behaviour, length.

In [ ]:
for name, path in [('sft', SFT), ('dpo_hh', DPO_HH), ('dpo_uf', DPO_UF)]:
    !python scripts/generate.py --model {path} --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/sanity/{name}.json --generator {name} --limit 3 --max_new_tokens 256
import json
for name in ['sft', 'dpo_hh', 'dpo_uf']:
    for r in json.load(open(f'outputs/sanity/{name}.json')):
        print(f'[{name}]', r['instruction'][:80], '\n ->', r['output'][:300], '\n')

Full generation, two jobs in parallel (one per GPU). Files are resumable: re-running skips finished rows.

In [ ]:
!(python scripts/generate.py --model {SFT} --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/sft_alpaca.json --generator sft --device cuda:0 && python scripts/generate.py --model {DPO_UF} --eval_set eval_sets/alpaca_eval_300_seed42.json --output outputs/generations/dpo_uf_alpaca.json --generator dpo_uf --device cuda:0) > gen_gpu0.log 2>&1 & (python scripts/generate.py --model {SFT} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/sft_hh.json --generator sft --device cuda:1 && python scripts/generate.py --model {DPO_HH} --eval_set eval_sets/hh_rlhf_eval_300_seed42.json --output outputs/generations/dpo_hh_hh.json --generator dpo_hh --device cuda:1) > gen_gpu1.log 2>&1; wait
!tail -2 gen_gpu0.log gen_gpu1.log

In [ ]:
!python scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --chat_template --eval_set eval_sets/persona_style_eval_40.json --output outputs/generations/persona_base.json --generator smollm2-360m-instruct
!python scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --adapter {LORA} --chat_template --eval_set eval_sets/persona_style_eval_40.json --output outputs/generations/persona_lora.json --generator smollm2-360m-lora-dpo

Blind pair files (A/B order randomized with seed 42; identities only in `meta`).

In [ ]:
!python scripts/build_pairs.py --candidate outputs/generations/dpo_uf_alpaca.json --baseline outputs/generations/sft_alpaca.json --output outputs/eval/dpo_uf_vs_sft_alpaca_pairs.jsonl
!python scripts/build_pairs.py --candidate outputs/generations/dpo_hh_hh.json --baseline outputs/generations/sft_hh.json --output outputs/eval/dpo_hh_vs_sft_hh_pairs.jsonl
!python scripts/build_pairs.py --candidate outputs/generations/persona_lora.json --baseline outputs/generations/persona_base.json --output outputs/eval/lora_vs_base_persona_pairs.jsonl

In [ ]:
# Small zip of everything except checkpoints (metrics, configs, plots, generations).
!cd outputs && zip -qr ../generation_outputs.zip . -x "*/best/*" "*/final/*" "*/adapter/*" "*/trainer/*" && ls -la ../generation_outputs.zip
print("Now click 'Save Version' (Save & Run All or Quick Save) so outputs persist and can be attached later.")